# Phase 2, Part 1: Evaluating Antoine's method, with evidence

**Question.** Why does Antoine's method (re-embed from scratch on every new entity, OCSVM, ADAZOR Z-test,
sphere rule) give alarms that are the same for every embedding model, and does it detect real drift?

The line-by-line audit of the code (12 problems, with small demonstrations) is in
`part1a_audit_antoine.ipynb`. This notebook shows the **evidence from the runs**.

| Section | Claim | Evidence |
|---|---|---|
| 1.1 | The sphere rule (almost) never fires | geometry + every ratio logged in Phase 1 + the sphere-size sweep |
| 1.2 | Embeddings retrained from scratch cannot be compared directly | controlled test: same graph with several seeds, an identical rerun, a grown graph, a warm start; literature |
| 1.3 | The alarms come from re-embedding | the same method with the embedding held fixed, or aligned |
| 1.4 | The method fires when nothing changes | streams with no drift |
| 1.5 | Its statistics are biased | reference vs new-triple outlier rates; fixing the statistics step by step |

All settings: `phase2/settings.py` (embedding as in Phase 1: 64 dims, 10 epochs, seed 42; window 250;
nu 0.1; alpha 0.025). Results are produced on the server by `phase2/run_phase2.sh`, which also executes
this notebook, so the outputs below are the real runs.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import results_helpers as R
pd.set_option('display.max_columns', 40, 'display.width', 220)
print('results folder:', R.RESULTS)
R.settings_summary()

## Hypotheses, written before the runs

| # | Hypothesis | Supported if | Refuted if |
|---|---|---|---|
| H1.1 | The sphere rule cannot fire in practice | every logged ratio is below 1; the rule only retrains once k is shrunk far below 1 | ratios near or above 1 occur |
| H1.2 | Two from-scratch trainings are not comparable in raw coordinates, but are after alignment | raw cosine between the same entity in two trainings is near 0 while aligned cosine / neighbour overlap are clearly higher; an OCSVM applied to a retrained embedding flags far more than nu with no drift | raw coordinates agree (cosine near 1) |
| H1.3 | Re-embedding causes most alarms | the fixed-embedding (or aligned) version has clearly fewer alarms than Antoine's on the same stream | similar alarm counts |
| H1.4 | Antoine's method raises false alarms with no drift | most no-drift streams get at least one alarm | no-drift streams stay mostly silent |
| H1.5 | The reference rate is biased low | outlier rate on new triples > reference rate in (almost) every run; a held-out reference removes most no-drift alarms | rates equal; no change |

## 1.1 The sphere rule (almost) never fires

**Why it can't fire, from the method itself.** The rule draws a box around all graph triple features and
retrains the OCSVM only if a new triple is further from the box centre than half the box diagonal
(ratio > k, k = 1). The furthest any point *inside* a box can be from its centre is a corner, exactly half
a diagonal away. So the ratio exceeds 1 only for a triple *outside* the box of all existing triples.
New triples are built from the same normalised vectors and fall inside. (Proof and demonstration:
`part1a_audit_antoine.ipynb`, problem 1.)

**Evidence from the runs.** Every ratio logged at a Phase 1 re-embed under Antoine's rule:

In [ ]:
R.sphere_ratios()

**The sphere-size sweep.** The radius was shrunk so that the rule would retrain on about 10%, 50% and 90%
of re-embeds (k chosen from the ratios above), plus "always retrain". If the alarms come from the
change of embedding, retraining the OCSVM more often should lower false alarms and re-embed-driven alarms.

In [ ]:
R.sphere_sweep()

## 1.2 Embeddings retrained from scratch cannot be compared directly

**Why, from the method itself.** A KG embedding is trained only to get *relationships* right (TransE:
subject + relation = object). Rotating every vector by the same amount leaves every distance, and so the
loss, unchanged. Training has no reason to prefer one orientation, and random initialisation, triple
order and negative sampling decide which one it lands in. DistMult, ComplEx and RotatE have their own
versions of this freedom (sign flips, coordinate swaps, phase rotations).

**What other researchers report** (short quotes; items marked [check against PDF] were read from arXiv HTML and must be checked against the published PDF before citing):

In [ ]:
R.literature()

**Controlled test (step 4).** Each model is trained on the cutoff graph G0 and on G1 = G0 + the pre-switch
triples (more data, **no topic change**). Pairs compared, entities matched by label:

| Pair | What it isolates |
|---|---|
| same graph, same seed, trained twice | is training even repeatable? |
| same graph, other seed | only randomness differs: the noise floor |
| grown graph, from scratch, same seed | **exactly what Antoine's method does at every re-embed** |
| grown graph, warm start | training continued from the old model instead of from zero |
| grown graph, other seed | data and randomness differ |

*Raw cosine* compares coordinates directly. *Aligned cosine* first rotates one embedding onto the other
(orthogonal Procrustes, the method of Hamilton et al. 2016). *Shared nearest neighbours* ignores rotation
altogether.

In [ ]:
R.seed_pairs()

**What this does to the OCSVM, with no drift at all.** An OCSVM is fitted on Antoine's reference in the
old embedding and applied to new pre-switch triples: in the same embedding, in the retrained embedding as
it is (what Antoine's method does), and in the retrained embedding rotated back. The gap between
"same embedding" and "new embedding" is the outlier rate that re-embedding alone creates.

In [ ]:
R.seed_ocsvm()

## 1.3 The alarms come from re-embedding

Antoine's method on the real streams, in three versions that differ **only** in how the embedding is
handled: retrained from scratch (Antoine; Phase 1 runs), retrained and then aligned onto the previous
embedding, and trained once and held fixed (new entities get a folded-in vector). Same reference, OCSVM,
Z-test, sphere rule and refill rule.

In [ ]:
R.stream_real()

## 1.4 Antoine's method fires when nothing changes

Streams of pre-switch triples only (one topic) in random order: every alarm is false. The same streams
are used for all three versions.

In [ ]:
R.stream_no_drift()

## 1.5 Antoine's statistics are biased

**The reference is checked on its own training data.** The OCSVM's reference outlier rate is measured on
the triples it was trained on (about nu = 10% of them lie outside its boundary by construction); fresh
triples from the same distribution fall outside more often. If so, the outlier rate on new triples should
exceed the reference rate in nearly every run:

In [ ]:
R.insample_bias()

**Fixing the statistics one step at a time** (same fixed embedding and OCSVM; no-drift and injected-switch
streams, 100 each): Antoine's statistics -> a fair reference (rate measured on held-out triples,
conformal p-values, Bates et al. 2023) -> plus a threshold set for repeated tests on overlapping windows
(Cobb et al. 2022).

In [ ]:
R.statistics_steps()

## What is not covered here

- Gradual or subtle drift, and no-drift streams in real time order: next steps, if time allows.
- The methods that retrain are run on fewer models and repeats than the cheap ones (a training per new
  entity); see the `streams` column in each table.